In [42]:
import glob
import os.path as op
import pickle
import numpy as np
import matplotlib.pyplot as plt
import warnings

from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.linear_model import LogisticRegression as LOR
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA
from generalized_contrastive_PCA import gcPCA
from scipy.stats import median_abs_deviation as MAD

plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
})

## Define Helpers and Functions

In [3]:
def reject_outliers(X, m=3, scaling=False):
    """
    Adapted from https://medium.com/@saraswatp/exploring-data-anomalies-rejecting-outliers-with-python-660b1ed6bca6
    Returns a boolean mask where True = 'Keep' and False = 'Outlier'.
    """
    median = np.median(X, axis=0)
    # mad = MAD(X, axis=0, scale='normal')

    if scaling:
        mad = np.median(np.abs(X - median), axis=0) * 1.4826
    else:
        mad = np.median(np.abs(X - median), axis=0)
    
    upper_bound = median + m * mad
    lower_bound = median - m * mad

    mask = (X >= lower_bound) & (X <= upper_bound)

    return mask


def reject_outliers_by_subject_channel(burst_dict, value_key='peak_amp_base', m=3, scaling=False):
    """Compute outlier mask separately within each subject x chennel group.

    Args:
        burst_dict (dictionary): Burst dictionary containing at least:
            - 'subject'
            - 'channel'
            - value_key
        
        value_key (str, optional): Name of the feature on which to reject outliers. Defaults to 'peak_amp_base'.
        m (float, optional): Number of MADs from the median used as the cutoff. Defaults to 3.
        scaling (bool, optional): If True, scale MAD by 1.4826. Defaults to False.
    
    Returns:
        group_mask: ndarray of bool, shape (n_bursts,). Global mask with True = keep, False = outlier.
    """

    values = np.asarray(burst_dict[value_key])
    subjects_arr = np.asarray(burst_dict['subject'])
    channels_arr = np.asarray(burst_dict['channel'])

    n = len(values)
    if not (len(subjects_arr) == n and len(channels_arr) == n):
        raise ValueError(f"'subject', 'channel', and '{value_key}' must have the same length." )
    
    group_mask = np.ones(n, dtype=bool)
    unique_subjects = np.unique(subjects_arr)

    for sub in unique_subjects:
        sub_idx = subjects_arr == sub
        sub_channels = np.unique(channels_arr[sub_idx])

        for ch in sub_channels:
            idx = sub_idx & (channels_arr == ch)

            # Reject within this subject x channel subset
            local_mask = reject_outliers(values[idx], m=m, scaling=scaling)

            # Write local decisions back into full-length mask
            group_mask[idx] = local_mask

    return group_mask

In [4]:
def scaler_fit(X):
    median = np.median(X, axis=0)
    mad = MAD(X, axis=0, scale='normal')
    return median, mad


def scaler_transform(X, median, mad):
    return (X - median) / mad

In [24]:
def get_features_and_labels(PC_scores, bursts_dict, type_subs, cols_idx, n_bins, area, subject_idx):
    """
    Function to build the feature matrix and label vector for the logistic regression classifier.
    It takes as input the PC scores for all bursts, the bursts dictionary, the indices of the PCs
    to consider, the number of bins to use for each PC, the number of MADs to use for outlier
    rejection when building trial-level features, and the area to consider (STN or cortical).
    """
    
    trials = np.asarray(type_subs)
    X = np.zeros((len(trials), len(cols_idx) * n_bins))
    y = np.zeros(len(trials))
    r = 0

    subject_bursts_idx = (bursts_dict['subject'] == subject_idx) & area
    pc_bin_lims = np.zeros((len(cols_idx), n_bins + 1))
    for col_idx, col in enumerate(cols_idx):
        train_burst_idx = subject_bursts_idx & np.isin(bursts_dict['trial'], n_bins)
        pc_bin_lims[col_idx, :] = np.percentile(PC_scores[train_burst_idx, col], np.linspace(0, 100, n_bins + 1))

    # Build trial-level features for the requested trial subset
    trials = np.asarray(type_subs)
    for trial in trials:
        burst_idx = area & (bursts_dict['trial'] == trial) & (bursts_dict['subject'] == subject_idx)
        label = np.unique(bursts_dict['med'][burst_idx])[0]
        
        row = np.zeros(len(cols_idx) * n_bins, dtype=float)  # Pre-allocate row with zeros for PC-bin combinations with no bursts
        for col_idx, col in enumerate(cols_idx):
            for bin in range(n_bins):
                lower = pc_bin_lims[col_idx, bin]
                higher = pc_bin_lims[col_idx, bin+1]
                count = np.sum((PC_scores[burst_idx, col] >= lower) &
                                (PC_scores[burst_idx, col] < higher))
                row[col_idx * n_bins + bin] = count
        X[r, :] = row
        y[r] = 0 if label == 'OFF' else 1
        r += 1

    X = X[:r]
    y = y[:r]

    return X, y

## Data loading

### Channel labels

In [6]:
ch_labels = ['STN_L1', 'STN_L2', 'STN_L3', 'STN_L4', 'STN_R1', 'STN_R2', 'STN_R3', 'STN_R4', 'F3', 'Fz', 'F4', 'C3', 'C4', 'Cz']

### Subject labels

In [7]:
data_dir = '../../data/preprocessed_data/'
out_path = '../../data/derivatives/'
data_files = glob.glob(op.join(data_dir, 'dataClean_step1_*_MED_ON.mat'))
subjects = sorted([x.split('/')[-1].split('_')[3] for x in data_files])
print(subjects)

['s02', 's04', 's05', 's06', 's07', 's08', 's10', 's15', 's16', 's17', 's18']


### Aggregating burst features of the STN and of motor cortex (C3,C4) across all subjects

In [8]:
all_bursts={
    'subject': [],
    'med': [],
    'channel': [],
    'trial': [],
    'waveform': np.zeros((0,132)),
    'waveform_times': [],
    'peak_freq': [],
    'peak_amp_iter': [],
    'peak_amp_base': [],
    'peak_time': [],
    'peak_adjustment': [],
    'fwhm_freq': [],
    'fwhm_time': [],
    'polarity': [],
}

for subject in subjects:
    fname = op.join(out_path, 'beta', f'bursts_{subject}.pickle')
    if op.exists(fname):
        print(subject)
        with open(fname,'rb') as file:  # rb = read binary
            bursts = pickle.load(file)
        
        # stn_c_bursts = boolean containing True for STNs and C3, C4 and False for every other channels
        stn_c_bursts = np.char.startswith(bursts['channel'].astype(str), 'STN') | (bursts['channel']=='C3') | (bursts['channel']=='C4')
        
        # Loads the data from bursts to all_bursts
        for key in bursts.keys():
            if key=='waveform_times':
                all_bursts[key] = bursts[key]

            elif key=='waveform':
                all_bursts[key] = np.vstack([all_bursts[key], bursts[key][stn_c_bursts, :]])

            else:
                all_bursts[key] = np.hstack([all_bursts[key], bursts[key][stn_c_bursts]])

s02
s04
s05
s06
s07
s08
s10
s15
s16
s17
s18


## Perfom light data preprocessing
* Redefine the trial indices ensuring they're not overlapping between ON and OFF medication - subject-level based
* Convert burst time to ms
* Reject outliers based on their peak amplitude
* Scale the waveform

In [9]:
for subject in subjects:
    subj_idx = (all_bursts['subject']==subject)
    off_med_trials = np.unique(all_bursts['trial'][subj_idx & (all_bursts['med']=='OFF')])
    max_off_trial = np.max(off_med_trials)
    on_med_idx = subj_idx & (all_bursts['med']=='ON')
    all_bursts['trial'][on_med_idx] = all_bursts['trial'][on_med_idx] + max_off_trial + 1 

In [10]:
burst_times = all_bursts['waveform_times'] * 1000 

In [11]:
outlier_mask = reject_outliers_by_subject_channel(
    all_bursts,
    value_key='peak_amp_base',
    m=3,
    scaling=False
    )

len_mask = len(outlier_mask)

correct_bursts = {
    key: (value[outlier_mask] if len(value) == len_mask else value)
    for key, value in all_bursts.items()
    }

# print(f"Original trials: {len(all_bursts['peak_amp_base'])}")
# print(f"Corrected trials: {len(correct_bursts['peak_amp_base'])}")
# print(f"Removed trials: {len(all_bursts['peak_amp_base']) - len(correct_bursts['peak_amp_base'])}")

### Select bursts corresponding to C3/C4 and all STN contacts

In [12]:
c_idx = (correct_bursts['channel']=='C3') | (correct_bursts['channel']=='C4')
stn_idx = [ch.startswith('STN') for ch in correct_bursts['channel']]

In [13]:
on_idx = (correct_bursts['med']=='ON')
off_idx = (correct_bursts['med']=='OFF')
on_c_idx = c_idx & on_idx
off_c_idx = c_idx & off_idx
on_stn_idx = stn_idx & on_idx
off_stn_idx = stn_idx & off_idx

In [48]:
"""
```scaling_method``` accepts: 
    - 'MAD-based',
    - 'Robust',
    - 'Standard'
"""
scaling_method = 'MAD-based'

area = stn_idx
scaled_waveforms = np.zeros_like(correct_bursts['waveform'])
for subject in np.unique(correct_bursts['subject']):
    subj_idx = (correct_bursts['subject'] == subject)

    if scaling_method == 'Robust':
        """Perform a modified z-score based on median and IQR"""
        scaler = RobustScaler().fit(correct_bursts['waveform'][subj_idx & area])                # .fit() returns the median and IQR values
        scaled_waveforms[subj_idx, :] = scaler.transform(correct_bursts['waveform'][subj_idx])  # .transform() does the maths.

    elif scaling_method == 'Standard':
        """Perform a z-score based on mean and STD"""
        scaler = StandardScaler().fit(correct_bursts['waveform'][subj_idx & area])              # .fit() returns the mean and STD values
        scaled_waveforms[subj_idx, :] = scaler.transform(correct_bursts['waveform'][subj_idx])  # .transform() does the maths.

    elif scaling_method == 'MAD-based':
        """Perfom a modified z-score based on median and MAD"""
        median, mad = scaler_fit(correct_bursts['waveform'][subj_idx & area])                   # Scaler based on MAD
        scaled_waveforms[subj_idx, :] = scaler_transform(correct_bursts['waveform'][subj_idx], median, mad)

    else:
        raise AttributeError("Wrong scaling method. Valide options are ['MAD-based', 'Robust', 'Standard'].")

### Subject-level cross validation: split trials into train/test groups
* 1/ Perfom both rPCA and gcPCA
* 2/ Each PC is binned
* 3/ Compute unique logit on each PCA, taking number of PCs and bins as features

In [49]:
"""Parameters definition"""
subjects = np.array(subjects)
warnings.filterwarnings('ignore', category=UserWarning, module='generalized_contrastive_PCA')  # Suppress generalized_contrastive_PCA warnings

n_splits = 10
kf = KFold(n_splits=n_splits, shuffle=True, random_state=6)
n_bins = np.arange(1, 15, 1)     # Number of bins to divide PCs into
cols_rPC = np.arange(2, 21, 2)   # List of rPC axes to consider
n_gcPCs = np.arange(1, 11, 1)    # Half number of gcPCs to consider - Un/comment to deal with all or specific number of gcPCs

for n_gcPC, n_rPC in zip(n_gcPCs, cols_rPC):
    """Run multiple cross-validation logits on PCA outputs for different number of PCs"""
    
    # Un/comment to deal with all or specific number of gcPCs
    null_and_positive = np.arange(0, n_gcPC, 1)
    negative = np.arange(-n_gcPC, 0, 1)
    cols_gcPC = np.concatenate([null_and_positive, negative])

    n_rPC = np.arange(n_rPC)
    print("gcPCs:", cols_gcPC, len(cols_gcPC))
    print("rPCs:", n_rPC, len(n_rPC))

    n_subjects = len(subjects)
    n_cases = len(n_bins) * n_subjects
    n_rPCA_components = len(n_rPC)
    n_gcPCA_components = len(cols_gcPC)

    rPCA_results = {
        'accuracy': np.zeros(n_cases),                              # 1 mean accuracy across folds for each subject x n_bins combination
        'variance': np.zeros(n_cases),                              # 1 std accuracy across folds for each subject x n_bins combination
        'intercept': np.zeros(n_cases),                             # 1 mean intercept across folds for each subject x n_bins combination
        'mean_coefficients': np.zeros(n_cases, dtype=object),       # 1 array of mean coefficients across folds for each subject x n_bins combination
        'var_coefficients': np.zeros(n_cases, dtype=object),        # 1 array of std coefficients across folds for each subject x n_bins combination
        'score': np.zeros((n_cases, n_rPCA_components)),            # 1 array of mean PC scores across folds and bursts for each subject x n_bins combination
        'var_score': np.zeros((n_cases, n_rPCA_components)),
    }

    gcPCA_results = {                                               # Replicate the logic for gcPCA results
        'accuracy': np.zeros(n_cases),
        'variance': np.zeros(n_cases),
        'intercept': np.zeros(n_cases),
        'mean_coefficients': np.zeros(n_cases, dtype=object),
        'var_coefficients': np.zeros(n_cases, dtype=object),
        'score': np.zeros((n_cases, n_gcPCA_components)),
        'var_score': np.zeros((n_cases, n_gcPCA_components)),
    }

    metadata = {
        'n_bins': n_bins,
        'cols': len(cols_gcPC)*2,
        'Parameters': [
            "Based on MAD-scaled waveforms",
            "Outlier rejection with m=3 MADs",
            "Logistic regression classifier with default parameters but with increased maximum iterations to 10_000",
            "No X-row normalization",
            "No X scaling before classification"
        ]
    }
    np.save("./Subject-Level_outputs/outputs-1/metadata.npy", metadata)

    case_idx = 0  # index for each subject x n_bins combination in the results dictionaries

    for n_bin in n_bins:
        """Run multiple cross-validation logits on PCA outputs for different number of bins"""

        print(f'Bins: {n_bin}/{n_bins}')

        for subject in subjects:
            subject_idx = (correct_bursts['subject'] == subject) & area
            trials = np.unique(correct_bursts['trial'][subject_idx])

            rPCA_fold_accuracies = np.zeros(n_splits)
            gcPCA_fold_accuracies = np.zeros(n_splits)

            rPCA_fold_intercept = np.zeros(n_splits)
            gcPCA_fold_intercept = np.zeros(n_splits)

            rPCA_fold_coeffs = None
            gcPCA_fold_coeffs = None

            rPCA_fold_waveforms = np.zeros(n_splits, dtype=object)
            gcPCA_fold_waveforms = np.zeros(n_splits, dtype=object)

            rPCA_fold_scores = None
            gcPCA_fold_scores = None

            for fold_idx, (train_idx, test_idx) in enumerate(kf.split(trials)):
                """Run cross-validation logits on PCA ouptuts"""

                # Select bursts from training trials
                train_bursts_idx = np.isin(correct_bursts['trial'], trials[train_idx]) & subject_idx
                train_bursts_on_idx = train_bursts_idx & (correct_bursts['med'] == 'ON')
                train_bursts_off_idx = train_bursts_idx & (correct_bursts['med'] == 'OFF')

                # Fit rPCA on training trials' bursts
                rPCA_model = PCA(n_components=20, random_state=6)
                rPCA_model.fit(scaled_waveforms[train_bursts_idx, :])
                all_rPCA_scores = rPCA_model.transform(scaled_waveforms)
                #-----------------------------------------------------------------------------------
                # Fit gcPCA on training trials' bursts
                gcPCA_model = gcPCA(method='v4', normalize_flag=False)
                gcPCA_model.fit(scaled_waveforms[train_bursts_off_idx, :], scaled_waveforms[train_bursts_on_idx, :])
                all_gcPCA_scores = scaled_waveforms @ gcPCA_model.loadings_

                """To un/comment if you want to use all gcPCs:"""
                # n_gcPC = gcPCA_model.loadings_.shape[1]
                # cols_gcPC = np.arange(0, n_gcPC, 1)

                """Get features and labels for logit from PCA outputs"""
                X_rPCA_train, y_rPCA_train = get_features_and_labels(all_rPCA_scores, correct_bursts, trials[train_idx], n_rPC, n_bin, area, subject)
                X_rPCA_test, y_rPCA_test = get_features_and_labels(all_rPCA_scores, correct_bursts, trials[test_idx], n_rPC, n_bin, area, subject)
                #-----------------------------------------------------------------------------------
                X_gcPCA_train, y_gcPCA_train = get_features_and_labels(all_gcPCA_scores, correct_bursts, trials[train_idx], cols_gcPC, n_bin, area, subject)
                X_gcPCA_test, y_gcPCA_test = get_features_and_labels(all_gcPCA_scores, correct_bursts, trials[test_idx], cols_gcPC, n_bin, area, subject)

                """Logistic regression on PCA outputs with increased maximum iterations"""
                model_rPCA = LOR(max_iter=10_000).fit(X_rPCA_train, y_rPCA_train)
                y_pred_rPCA = model_rPCA.predict(X_rPCA_test)
                #-----------------------------------------------------------------------------------
                model_gcPCA = LOR(max_iter=10_000).fit(X_gcPCA_train, y_gcPCA_train)
                y_pred_gcPCA = model_gcPCA.predict(X_gcPCA_test)

                if rPCA_fold_coeffs is None:
                    n_rPCA_features = model_rPCA.coef_[0].shape[0]
                    n_gcPCA_features = model_gcPCA.coef_[0].shape[0]
                    rPCA_fold_coeffs = np.zeros((n_splits, n_rPCA_features))
                    gcPCA_fold_coeffs = np.zeros((n_splits, n_gcPCA_features))

                """Classify and store waveforms to PC scores"""
                rPCA_fold_waveforms[fold_idx] = (X_rPCA_train, X_rPCA_test)
                gcPCA_fold_waveforms[fold_idx] = (X_gcPCA_train, X_gcPCA_test)

                """Store logit intercept and coefficients accordingly"""
                rPCA_fold_intercept[fold_idx] = model_rPCA.intercept_[0]
                gcPCA_fold_intercept[fold_idx] = model_gcPCA.intercept_[0]
                rPCA_fold_coeffs[fold_idx] = model_rPCA.coef_[0]
                gcPCA_fold_coeffs[fold_idx] = model_gcPCA.coef_[0]

                """Store logit accuracies accordingly"""
                rPCA_fold_accuracies[fold_idx] = np.mean(y_rPCA_test == y_pred_rPCA)
                gcPCA_fold_accuracies[fold_idx] = np.mean(y_gcPCA_test == y_pred_gcPCA)
                print(f'Fold {fold_idx}: gcPCA accuracy = {gcPCA_fold_accuracies[fold_idx]:.3f}, rPCA accuracy = {rPCA_fold_accuracies[fold_idx]:.3f}')

                if fold_idx == 0:                    
                    rPCA_fold_scores = np.zeros((n_splits, scaled_waveforms.shape[0], n_rPCA_components))
                    gcPCA_fold_scores = np.zeros((n_splits, scaled_waveforms.shape[0], n_gcPCA_components))

                rPCA_fold_scores[fold_idx] = all_rPCA_scores[:, n_rPC]
                gcPCA_fold_scores[fold_idx] = all_gcPCA_scores[:, cols_gcPC]

            for pca_type, results, fold_accuracies, fold_intercept, fold_coeffs, fold_scores in [
                ('rPCA', rPCA_results, rPCA_fold_accuracies, rPCA_fold_intercept, rPCA_fold_coeffs, rPCA_fold_scores),
                ('gcPCA', gcPCA_results, gcPCA_fold_accuracies, gcPCA_fold_intercept, gcPCA_fold_coeffs, gcPCA_fold_scores),
            ]:
                results['accuracy'][case_idx] = np.mean(fold_accuracies)
                results['variance'][case_idx] = np.std(fold_accuracies)
                results['intercept'][case_idx] = np.mean(fold_intercept)
                results['mean_coefficients'][case_idx] = np.mean(fold_coeffs, axis=0)
                results['var_coefficients'][case_idx] = np.std(fold_coeffs, axis=0)
                results['score'][case_idx] = np.mean(fold_scores, axis=(0, 1))
                results['var_score'][case_idx] = np.std(fold_scores, axis=(0, 1))

            case_idx += 1

    """Data saving"""

    gcPC_file = op.join(f'./Subject-Level_outputs/outputs-1/gcPCA_outputs_{len(cols_gcPC)}_stn.npy')
    rPC_file = op.join(f'./Subject-Level_outputs/outputs-1/rPCA_outputs_{len(n_rPC)}_stn.npy')

    if op.exists(gcPC_file):
        print(f"File {gcPC_file} already exists. Skipping saving.")
    else:
        print(f"Saving file {gcPC_file}...")
        np.save(f'./Subject-Level_outputs/outputs-1/gcPCA_outputs_{len(cols_gcPC)}_stn', gcPCA_results)
        if op.exists(gcPC_file):
            print(f"File {gcPC_file} successfully saved.")
        else:
            print(f"Error: File {gcPC_file} was not found.")

    if op.exists(rPC_file):
        print(f"File {rPC_file} already exists. Skipping saving.")
    else:
        print(f"Saving file {rPC_file}...")
        np.save(f'./Subject-Level_outputs/outputs-1/rPCA_outputs_{len(n_rPC)}_stn', rPCA_results)
        if op.exists(rPC_file):
            print(f"File {rPC_file} successfully saved.")
        else:
            print(f"Error: File {rPC_file} was not found.")

    print(f"{len(cols_gcPC)}/{n_gcPCs*2}")
    print("_____________________________\n")

gcPCs: [ 0 -1] 2
rPCs: [0 1] 2
Bins: 1/[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14]
Fold 0: gcPCA accuracy = 1.000, rPCA accuracy = 0.500
Fold 1: gcPCA accuracy = 0.750, rPCA accuracy = 1.000
Fold 2: gcPCA accuracy = 0.250, rPCA accuracy = 0.500
Fold 3: gcPCA accuracy = 1.000, rPCA accuracy = 0.500
Fold 4: gcPCA accuracy = 0.500, rPCA accuracy = 0.500
Fold 5: gcPCA accuracy = 0.250, rPCA accuracy = 0.250
Fold 6: gcPCA accuracy = 0.750, rPCA accuracy = 0.750
Fold 7: gcPCA accuracy = 0.750, rPCA accuracy = 0.250
Fold 8: gcPCA accuracy = 0.667, rPCA accuracy = 0.333
Fold 9: gcPCA accuracy = 1.000, rPCA accuracy = 0.667
Fold 0: gcPCA accuracy = 0.500, rPCA accuracy = 1.000
Fold 1: gcPCA accuracy = 1.000, rPCA accuracy = 0.750
Fold 2: gcPCA accuracy = 0.750, rPCA accuracy = 0.750
Fold 3: gcPCA accuracy = 0.750, rPCA accuracy = 0.500
Fold 4: gcPCA accuracy = 1.000, rPCA accuracy = 0.500
Fold 5: gcPCA accuracy = 1.000, rPCA accuracy = 0.500
Fold 6: gcPCA accuracy = 0.750, rPCA accuracy = 0.750